# Setup and manual configuration
Mounting Google Drive, cloning the repository from GitHub, setting up input/output paths, and installing specific dependencies inline.
Modify the variables in this cell before starting a new experiment.

In [ ]:
import os
from google.colab import drive

# Mount Google Drive for datasets and results
drive.mount('/content/drive')

# ==========================================
# --- MANUAL EXPERIMENT CONFIGURATION ---
# ==========================================
GIT_REPO_URL = 'https://github.com/EmanuelePietroCometti/SuperSimpleNet.git'
REPO_PATH = '/content/SuperSimpleNet'
DATA_PATH = '/content/drive/MyDrive/Tesi/MVTec'
SAVE_PATH = '/content/drive/MyDrive/Tesi/Risultati_SSN/run_custom_01_no_dust'
CATEGORY = 'custom_no_dust'
SETUP_NAME = 'superSimpleNet_baseline'
MODE = 'sup'
# ==========================================

# Clone or Update the repository from GitHub
if not os.path.exists(REPO_PATH):
    print(">>> Cloning repository from GitHub...")
    # Aggiunto REPO_PATH al clone per forzare la cartella di destinazione corretta
    !git clone {GIT_REPO_URL} {REPO_PATH}
else:
    print(">>> Repository already cloned. Pulling latest changes...")
    os.chdir(REPO_PATH)
    !git pull

# Change to the working directory
os.chdir(REPO_PATH)
print(f"Working directory set to: {os.getcwd()}")

# Directly create the output folder on Drive
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"Output folder ready on Drive: {SAVE_PATH}")

# Install standard dependencies
!pip install tqdm numpy==1.26.0 anomalib==0.7

# Install specific PyTorch version with CUDA 11.8 support
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 --extra-index-url https://download.pytorch.org/whl/cu118

# Optional: Install wandb for experiment tracking
!pip install wandb

# Hyperparameter Optimization (Optional)
Launch the fine-tuning script. It dynamically reads the parameters configured in Cell 1.

In [ ]:
import os
os.chdir(REPO_PATH)

print(f">>> Starting Hyperparameter Optimization on dataset: {CATEGORY}...")

!python hyperparameter_finetuning.py \
    --dataset mvtec \
    --category {CATEGORY} \
    --data_path {DATA_PATH} \
    --datasets_folder {DATA_PATH} \
    --results_save_path {SAVE_PATH} \
    --setup_name {SETUP_NAME} \
    --epochs 100 \
    --batch 4

# Training, Evaluation and ONNX Export
This single cell orchestrates the entire pipeline:
1. Runs `train.py` with the specified parameters.
2. Automatically locates the generated `.pt` (or `.ckpt`) weights file on Google Drive.
3. Passes the weights file to `eval.py` and `export_onnx.py`.
4. Organizes the generated ONNX models into a dedicated subfolder.

In [ ]:
import os
import glob
import shutil

os.chdir(REPO_PATH)

print(f">>> Starting Training on {CATEGORY}...")
print(f">>> Saving results directly to: {SAVE_PATH}\n")

# Lancia train.py iniettando le variabili Python
!python train.py \
    --dataset mvtec \
    --category {CATEGORY} \
    --mode {MODE} \
    --data_path {DATA_PATH} \
    --datasets_folder {DATA_PATH} \
    --results_save_path {SAVE_PATH} \
    --setup_name {SETUP_NAME} \
    --backbone wide_resnet50_2 \
    --layers layer2 layer3 \
    --image_size 512 512 \
    --epochs 100 \
    --batch 4 \
    --perlin_thr 0.2 \
    --noise_std 0.015 \
    --seg_lr 0.0002 \
    --dec_lr 0.0002 \
    --adapt_lr 0.0001 \
    --patch_size 3 \
    --gamma 0.4 \
    --eval_step_size 5

print(f"\n>>> Searching for the generated weights in {SAVE_PATH}...")
# Cerca ricorsivamente qualsiasi file dei pesi generato dal training
weight_files = glob.glob(os.path.join(SAVE_PATH, "**", "*.pt"), recursive=True) + \
               glob.glob(os.path.join(SAVE_PATH, "**", "*.pth"), recursive=True) + \
               glob.glob(os.path.join(SAVE_PATH, "**", "*.ckpt"), recursive=True)

if not weight_files:
    print("[ERROR] No weights file found. The training process might have failed.")
else:
    # Prende il primo file dei pesi trovato
    WEIGHTS_FILE = weight_files[0]
    print(f">>> Found weights file: {WEIGHTS_FILE}")
    
    print("\n" + "="*40)
    print("--- STARTING EVALUATION ---")
    print("="*40)
    !python eval.py "{WEIGHTS_FILE}" \
    --dataset mvtec \
    --category {CATEGORY} \
    --datasets_folder {DATA_PATH} \
    --results_save_path {SAVE_PATH} \
    --image_size 512 512 \
    --batch 4
    
    print("\n" + "="*40)
    print("--- STARTING ONNX EXPORT ---")
    print("="*40)
    !python export_onnx.py "{WEIGHTS_FILE}"
    
    print("\n>>> Reorganizing ONNX files...")
    # Crea una cartella 'onnx' e sposta i modelli esportati
    onnx_dir = os.path.join(SAVE_PATH, "onnx")
    os.makedirs(onnx_dir, exist_ok=True)
    
    for onnx_file in glob.glob(os.path.join(SAVE_PATH, "**", "*.onnx"), recursive=True):
        shutil.move(onnx_file, os.path.join(onnx_dir, os.path.basename(onnx_file)))
        
    print(f"\n>>> Pipeline completed successfully! ONNX models saved in: {onnx_dir}")